In [2]:
from pymoo.indicators.hv import HV

import numpy as np
import pickle
import statistics
import pandas as pd

# Hypervolume:

In [4]:
# Name of the experiment
experiment_names = ['wfg1', 'wfg3', 'wfg4', 'wfg5', 'wfg6', 'wfg7', 'wfg8', 'wfg9']
# Choosing the files
choose_global_best_attribution_type = [0] # [0, 1, 2, 3]
choose_Xr_pool_type = [1] # [0, 1, 2]
choose_DE_mutation_type = [0] # [0, 1, 2, 3, 4]

# Name of chosen files
file_names = [
        f"E{i+1}V{j+1}D{k+1}_{experiment_name}.pkl"
        for i in choose_global_best_attribution_type
        for j in choose_Xr_pool_type
        for k in choose_DE_mutation_type
        for experiment_name in experiment_names
    ]

file_names.extend(['NSGA2_' + name + '.pkl' for name in experiment_names])

num_dataset = len(file_names) # Number of dataset

# Take the results
results = []
for i in range(num_dataset):
  with open("../result/" + file_names[i], "rb") as f:
    results.append(pickle.load(f))

### Calculate the Hypervolume:

In [7]:
# Create the hypervolume indicator
ref_point = np.array([10]*3)
indicator = HV(ref_point=ref_point)

# Create the pandas DataFrame columns
columns = ['Algorithm', 'Function', 'Mean HV', 'Std. Dev.', 'Min. HV', 'Max. HV', 'Combined HV']
# Store the datas
data = []
for i, fn in enumerate(file_names):
  HVs = []
  r = results[i]
  for j in range(len(results[i])-1):
    HVs.append(indicator(r[j+1]["F"]))
  if len(HVs) > 1:
    exp = fn.split('_', 1)
    alg = exp[0]
    func = (exp[1].split('.', 1))[0]
    data.append([alg, func.upper(), statistics.mean(HVs), statistics.stdev(HVs), min(HVs), max(HVs), indicator(results[i]['combined'][1])])
  else:
    exp = fn.split('_', 1)
    alg = exp[0]
    func = (exp[1].split('.', 1))[0]
    data.append([alg, func.upper(), statistics.mean(HVs), 0, min(HVs), max(HVs), indicator(results[i]['combined'][1])])

# Create the DataFrame
df = pd.DataFrame(data, columns=columns)
df

,Algorithm,Function,Mean HV,Std. Dev.,Min. HV,Max. HV,Combined HV
0,E1V2D1,WFG1,737.139103,1.450353,733.250548,739.472590,743.699255
1,E1V2D1,WFG3,900.798793,3.467188,894.157601,908.349940,911.320379
2,E1V2D1,WFG4,950.244446,1.893958,946.160729,953.361793,958.796857
3,E1V2D1,WFG5,944.576445,2.085518,941.040651,948.332445,952.030880
4,E1V2D1,WFG6,925.539859,8.611791,910.619092,941.308002,949.586430
5,E1V2D1,WFG7,953.474079,2.018590,948.703035,957.932332,963.149891
6,E1V2D1,WFG8,927.378407,2.700521,922.228295,932.698335,940.268148
7,E1V2D1,WFG9,929.553481,15.233590,897.690334,945.870894,952.870519
8,NSGA2,WFG1,732.555977,10.825517,719.211273,751.724446,757.384522
9,NSGA2,WFG3,913.490528,1.299070,910.618530,916.011619,917.757030


In [9]:
print(df.to_latex(index=False, float_format="%.3f", escape=False, column_format='lcccccc', label='tab:hv', caption='Hypervolume indicator of the algorithms on the test problems. The best results are highlighted in bold font.'))

\begin{table}
\caption{Hypervolume indicator of the algorithms on the test problems. The best results are highlighted in bold font.}
\label{tab:hv}
\begin{tabular}{lcccccc}
\toprule
Algorithm & Function & Mean HV & Std. Dev. & Min. HV & Max. HV & Combined HV \\
\midrule
E1V2D1 & WFG1 & 737.139 & 1.450 & 733.251 & 739.473 & 743.699 \\
E1V2D1 & WFG3 & 900.799 & 3.467 & 894.158 & 908.350 & 911.320 \\
E1V2D1 & WFG4 & 950.244 & 1.894 & 946.161 & 953.362 & 958.797 \\
E1V2D1 & WFG5 & 944.576 & 2.086 & 941.041 & 948.332 & 952.031 \\
E1V2D1 & WFG6 & 925.540 & 8.612 & 910.619 & 941.308 & 949.586 \\
E1V2D1 & WFG7 & 953.474 & 2.019 & 948.703 & 957.932 & 963.150 \\
E1V2D1 & WFG8 & 927.378 & 2.701 & 922.228 & 932.698 & 940.268 \\
E1V2D1 & WFG9 & 929.553 & 15.234 & 897.690 & 945.871 & 952.871 \\
NSGA2 & WFG1 & 732.556 & 10.826 & 719.211 & 751.724 & 757.385 \\
NSGA2 & WFG3 & 913.491 & 1.299 & 910.619 & 916.012 & 917.757 \\
NSGA2 & WFG4 & 958.126 & 2.980 & 947.222 & 962.036 & 965.275 \\
NSGA2 & WFG5 & 